In [37]:
import pandas as pd
import numpy as np
from scipy.constants import pi

branch_data = pd.read_csv("branch.csv")
h_5 = 5 * 50

def create_z_5(branch_data):
    return [
        complex(row['r'], row['l'] * 2 * pi * h_5)
        for _, row in branch_data.iterrows()
    ]

def create_y_5(branch_data):
    return [
        complex(0, row['c'] * 2 * pi * h_5)
        for _, row in branch_data.iterrows()
    ]

def create_gammaL_5(branch_data, z_5, y_5):
    z_5 = np.array(z_5)
    y_5 = np.array(y_5)
    gammaL_5 = []
    for i, row in branch_data.iterrows():
        length = row['Length']
        gammaL = length * np.sqrt(z_5[i] * y_5[i])
        gammaL_5.append(gammaL)
    return gammaL_5

z_5 = create_z_5(branch_data)
y_5 = create_y_5(branch_data)
gammaL_5 = create_gammaL_5(branch_data, z_5, y_5)

def Z0_5(z_5, y_5):
    return np.sqrt(np.array(z_5) / np.array(y_5))

Z0_5 = Z0_5(z_5, y_5)
def ABCD_5(gammaL_5, Z0_5):
    A = np.cosh(gammaL_5)
    B = Z0_5 * np.sinh(gammaL_5)
    C = (1 / Z0_5) * np.sinh(gammaL_5)
    D = np.cosh(gammaL_5)
    return A, B, C, D
A_5, B_5, C_5, D_5 = ABCD_5(gammaL_5, Z0_5)
ABCD_5_df = pd.DataFrame({
    'From Bus Number': branch_data['From Bus  Number'],
    'To Bus Number'  : branch_data['To Bus  Number'],
    'A_5'           : A_5,
    'B_5'           : B_5,
    'C_5'           : C_5,
    'D_5'           : D_5
})

ABCD_5_df.to_csv("ABCD_5.csv", index=False)


In [38]:
print(z_5)
print(y_5)

[(0.027104+1.4737792332630633j), (0.027104+1.4737792332630633j), (0.040967143+1.638686144038972j), (0.040967143+1.638686144038972j), (0.038236+1.5294404011030405j), (0.038236+1.5294404011030405j), (0.076053405+1.9348220923443065j), (0.076053405+1.9348220923443065j), (0.037948173+1.5299870382247651j), (0.037948173+1.5299870382247651j), (0.038033475+1.5299210647790398j), (0.038033475+1.5299210647790398j), (0.075983717+1.9360001895894028j), (0.075983717+1.9360001895894028j), (0.0121+0.726000071096026j), (0.038070732+1.5302069497105162j), (0.037470968+1.5300640072447782j), (0.075988+1.9360001895894028j), (0.075988+1.9360001895894028j), (0.038236+1.5294404011030405j), (0.038236+1.5294404011030405j), (0.038236+1.5294404011030405j), (0.038236+1.5294404011030405j), (0.0518848+1.479104232810898j), (0.0518848+1.479104232810898j), (0.016456+1.0260802889779426j), (0.016456+1.0260802889779426j), (0.016491852+1.0271562844617972j), (0.038236+1.5294404011030405j), (0.038236+1.5294404011030405j), (0.03

In [39]:
print(ABCD_5_df)

    From Bus Number  To Bus Number                 A_5                   B_5  \
0              2135           2220  0.992559+0.000137j  0.620299+ 33.812836j   
1              2135           2220  0.992559+0.000137j  0.620299+ 33.812836j   
2              2135           2400  0.672626+0.007705j  4.476263+203.803171j   
3              2135           2400  0.672626+0.007705j  4.476263+203.803171j   
4              2135           2970  0.960158+0.000989j  1.898206+ 76.963414j   
5              2135           2970  0.960158+0.000989j  1.898206+ 76.963414j   
6              2220           2225  0.995062+0.000194j  1.402356+ 35.735361j   
7              2220           2225  0.995062+0.000194j  1.402356+ 35.735361j   
8              2220           2230  0.986053+0.000345j  1.131617+ 45.838437j   
9              2220           2230  0.986053+0.000345j  1.131617+ 45.838437j   
10             2220           2570  0.924292+0.001858j  2.545852+105.125171j   
11             2220           2570  0.92

In [40]:
import pandas as pd
import numpy as np

# Load and ensure numeric types
ABCD = pd.read_csv("ABCD_5.csv")

# Clean column names (optional, in case there are hidden spaces)
ABCD.columns = ABCD.columns.str.strip()

# Convert to complex numbers safely
for col in ['A_5', 'B_5', 'C_5', 'D_5']:
    ABCD[col] = ABCD[col].apply(lambda x: complex(x.replace('i', 'j')) if isinstance(x, str) else complex(x))

def combine_parallel(group):
    if len(group) == 1:
        row = group.iloc[0]
        return pd.Series({
            'From Bus Number': row['From Bus Number'],
            'To Bus Number': row['To Bus Number'],
            'A': row['A_5'],
            'B': row['B_5'],
            'C': row['C_5'],
            'D': row['D_5']
        })

    # Start with first line
    A_eq, B_eq, C_eq, D_eq = group.iloc[0][['A_5', 'B_5', 'C_5', 'D_5']]
    for _, row in group.iloc[1:].iterrows():
        A1, B1, C1, D1 = A_eq, B_eq, C_eq, D_eq
        A2, B2, C2, D2 = row['A_5'], row['B_5'], row['C_5'], row['D_5']

        A_eq = (A1 * B2 + A2 * B1) / (B1 + B2)
        B_eq = (B1 * B2) / (B1 + B2)
        C_eq = C1 + C2 + ((A1 - A2) * (D1 - D2)) / (B1 + B2)
        D_eq = (D1 * B2 + D2 * B1) / (B1 + B2)

    return pd.Series({
        'From Bus Number': group.iloc[0]['From Bus Number'],
        'To Bus Number': group.iloc[0]['To Bus Number'],
        'A': A_eq, 'B': B_eq, 'C': C_eq, 'D': D_eq
    })

# Combine lines with same (From, To)
ABCD_combined = (
    ABCD.groupby(['From Bus Number', 'To Bus Number'])
    .apply(combine_parallel)
    .reset_index(drop=True)
)

# Save result
ABCD_combined.to_csv("ABCD_parallel_combined.csv", index=False)

print(ABCD_combined.head())


   From Bus Number   To Bus Number                   A                     B  \
0   2135.0+   0.0j  2220.0+   0.0j  0.992559+0.000137j  0.310150+ 16.906418j   
1   2135.0+   0.0j  2400.0+   0.0j  0.672626+0.007705j  2.238131+101.901585j   
2   2135.0+   0.0j  2970.0+   0.0j  0.960158+0.000989j  0.949103+ 38.481707j   
3   2220.0+   0.0j  2225.0+   0.0j  0.995062+0.000194j  0.701178+ 17.867681j   
4   2220.0+   0.0j  2230.0+   0.0j  0.986053+0.000345j  0.565808+ 22.919218j   

                    C                   D  
0 -0.000000+0.000877j  0.992559+0.000137j  
1 -0.000016+0.005374j  0.672626+0.007705j  
2 -0.000001+0.002029j  0.960158+0.000989j  
3 -0.000000+0.000551j  0.995062+0.000194j  
4 -0.000000+0.001209j  0.986053+0.000345j  


C:\Users\User\AppData\Local\Temp\ipykernel_2940\3617034173.py:46: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(combine_parallel)


In [41]:
import pandas as pd
import numpy as np

# Example Z0 calculation (if you already have Z0_11 column)

Z0 = np.abs(Z0_5)  # or define a constant if same for all lines
Y0 = 1 / Z0

def abcd2s(ABCD_combined, Z0, Y0):
    S_data = []
    for _, row in ABCD_combined.iterrows():
        # Extract ABCD parameters
        AA = row["A"]
        BB = row["B"]
        CC = row["C"]
        DD = row["D"]

        # Compute S-parameters
        denom1 = (AA + BB * Y0 + CC * Z0 + DD)
        denom2 = (AA + BB * Y0 + CC * Z0 - DD)

        S11 = (AA + BB * Y0 - CC * Z0 - DD) / denom1
        S12 = 2 * (AA * DD - BB * CC) / denom1
        S21 = 2 / denom1
        S22 = (-AA + BB * Y0 - CC * Z0 + DD) / denom1

        

    return np.array([[S11, S12],[S21, S22]], dtype=complex)

# Run conversion
S_params = abcd2s(ABCD_combined, Z0, Y0)


print(S_params)

[[[-0.04488331-0.15880392j -0.04488331-0.15880392j
   -0.04403351-0.1567088j  -0.04403351-0.1567088j
   -0.04403362-0.15670906j -0.04403362-0.15670906j
   -0.07808471-0.22769582j -0.07808471-0.22769582j
   -0.04403851-0.1567212j  -0.04403851-0.1567212j
   -0.04404037-0.1567258j  -0.04404037-0.1567258j
   -0.07812507-0.2277672j  -0.07812507-0.2277672j
    0.12434405+0.28027227j -0.04406359-0.1567833j
   -0.0439792 -0.1565742j  -0.07812745-0.2277714j
   -0.07812745-0.2277714j  -0.04403362-0.15670906j
   -0.04403362-0.15670906j -0.04403362-0.15670906j
   -0.04403362-0.15670906j -0.0440206 -0.15667682j
   -0.0440206 -0.15667682j  0.09011475+0.23052198j
    0.09011475+0.23052198j  0.090028  +0.23037908j
   -0.04403362-0.15670906j -0.04403362-0.15670906j
   -0.04404104-0.15672744j -0.04404104-0.15672744j
   -0.07810409-0.22773009j -0.07810409-0.22773009j
   -0.0440323 -0.1567058j  -0.0440323 -0.1567058j
   -0.07808359-0.22769384j -0.07808359-0.22769384j
   -0.04403362-0.15670906j -0.04403362

In [42]:
import pandas as pd
import numpy as np

def connection_matrix(ABCD_combined):
    # Clean and extract unique buses
    buses = pd.unique(ABCD_combined[['From Bus Number', 'To Bus Number']].values.ravel())
    
    # Sort buses for readability (optional)
    buses = np.sort(buses)
    
    # Map bus numbers to index
    bus_index = {bus: idx for idx, bus in enumerate(buses)}
    n = len(buses)

    # Initialize connection matrix
    connection_matrix = np.zeros((n, n), dtype=int)

    # Build the adjacency (connection) matrix
    for _, row in ABCD_combined.iterrows():
        from_idx = bus_index[row['From Bus Number']]
        to_idx = bus_index[row['To Bus Number']]
        connection_matrix[from_idx, to_idx] = 1
        connection_matrix[to_idx, from_idx] = 1  # undirected

    # Create DataFrame with formatted bus names
    bus_labels = [f"{bus.real:+.1f}+{bus.imag:+.1f}j" if isinstance(bus, complex) else f"{bus:.1f}+0.0j" for bus in buses]
    connection_df = pd.DataFrame(connection_matrix, index=bus_labels, columns=bus_labels)
    return connection_df

# Create the connection matrix DataFrame
connection_matrix_df = connection_matrix(ABCD_combined)

# Convert to NumPy array
conn_np = connection_matrix_df.to_numpy()

print(conn_np)
print("Shape:", conn_np.shape)

[[0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 1]
 [1 0 0 1 1 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 1 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
 [0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
 [0 1 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 1 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0]
 [0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 1 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
 [0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0 0 0 1 0 1 0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 1 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
 [1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 1]
 [0 1 0 0 

In [43]:
def connection_matrix_with_labels(ABCD_combined):
    # Extract unique buses
    buses = pd.unique(ABCD_combined[['From Bus Number', 'To Bus Number']].values.ravel())
    buses = np.sort(buses)
    
    # Map bus numbers to simple labels
    from_labels = {bus: f"a{i+1}" for i, bus in enumerate(buses)}
    to_labels = {bus: f"b{i+1}" for i, bus in enumerate(buses)}
    
    # Map bus numbers to index
    bus_index = {bus: idx for idx, bus in enumerate(buses)}
    n = len(buses)
    
    # Initialize connection matrix
    conn_matrix = np.zeros((n, n), dtype=int)
    
    # Fill matrix
    for _, row in ABCD_combined.iterrows():
        from_idx = bus_index[row['From Bus Number']]
        to_idx = bus_index[row['To Bus Number']]
        conn_matrix[from_idx, to_idx] = 1
        conn_matrix[to_idx, from_idx] = 1  # undirected
    
    # Create DataFrame with new labels
    from_bus_labels = [from_labels[bus] for bus in buses]
    to_bus_labels = [to_labels[bus] for bus in buses]
    conn_df = pd.DataFrame(conn_matrix, index=from_bus_labels, columns=to_bus_labels)
    
    return conn_df

# Example usage:
connection_matrix_df = connection_matrix_with_labels(ABCD_combined)
print(connection_matrix_df)

     b1  b2  b3  b4  b5  b6  b7  b8  b9  b10  ...  b17  b18  b19  b20  b21  \
a1    0   1   0   0   0   0   0   0   0    0  ...    0    0    0    0    0   
a2    1   0   0   1   1   0   0   0   0    0  ...    0    0    1    0    1   
a3    0   0   0   0   0   0   0   0   0    0  ...    0    0    0    0    0   
a4    0   1   0   0   0   0   0   0   0    0  ...    0    0    0    0    0   
a5    0   1   0   0   0   1   0   0   0    0  ...    0    0    0    0    0   
a6    0   0   0   0   1   0   0   1   0    0  ...    0    0    0    0    0   
a7    0   0   0   0   0   0   0   0   1    0  ...    0    0    0    0    0   
a8    0   0   0   0   0   1   0   0   0    0  ...    0    0    0    0    0   
a9    0   0   0   0   0   0   1   0   0    1  ...    0    0    0    0    0   
a10   0   0   0   0   0   0   0   0   1    0  ...    0    0    0    0    0   
a11   0   0   1   0   0   0   0   0   0    0  ...    0    0    1    0    0   
a12   0   0   0   0   0   0   0   0   0    0  ...    0    0    0

In [44]:
buses = np.sort(pd.unique(ABCD_combined[['From Bus Number', 'To Bus Number']].values.ravel()))
for i, bus in enumerate(buses):
    print(f"a{i+1} / b{i+1} -> Bus {bus}")


a1 / b1 -> Bus (2135+0j)
a2 / b2 -> Bus (2220+0j)
a3 / b3 -> Bus (2222+0j)
a4 / b4 -> Bus (2225+0j)
a5 / b5 -> Bus (2230+0j)
a6 / b6 -> Bus (2240+0j)
a7 / b7 -> Bus (2245+0j)
a8 / b8 -> Bus (2250+0j)
a9 / b9 -> Bus (2280+0j)
a10 / b10 -> Bus (2281+0j)
a11 / b11 -> Bus (2300+0j)
a12 / b12 -> Bus (2305+0j)
a13 / b13 -> Bus (2306+0j)
a14 / b14 -> Bus (2307+0j)
a15 / b15 -> Bus (2350+0j)
a16 / b16 -> Bus (2400+0j)
a17 / b17 -> Bus (2560+0j)
a18 / b18 -> Bus (2561+0j)
a19 / b19 -> Bus (2570+0j)
a20 / b20 -> Bus (2580+0j)
a21 / b21 -> Bus (2691+0j)
a22 / b22 -> Bus (2705+0j)
a23 / b23 -> Bus (2810+0j)
a24 / b24 -> Bus (2815+0j)
a25 / b25 -> Bus (2830+0j)
a26 / b26 -> Bus (2970+0j)


In [ ]:
z_5 = np.array(z_5, dtype=complex).flatten()

In [46]:
conn_df = connection_matrix_df

# Convert to numpy matrix (still 0/1)
conn_np = conn_df.values

# Number of nodes
N = conn_np.shape[0]

# Map labels "a1..a26" back to numbers 1..26
node_labels = list(conn_df.index)     # ["a1", "a2", ..., "a26"]
node_numbers = [int(label[1:]) for label in node_labels]   # strip the "a"


In [47]:
lines = []

for i in range(N):
    for j in range(i+1, N):
        if conn_np[i, j] == 1:
            bus_i = node_numbers[i]
            bus_j = node_numbers[j]
            lines.append((bus_i, bus_j))

print(lines)


[(1, 2), (1, 16), (1, 26), (2, 4), (2, 5), (2, 19), (2, 21), (3, 11), (5, 6), (6, 8), (7, 9), (7, 22), (9, 10), (11, 19), (12, 13), (12, 15), (13, 20), (14, 20), (17, 19), (18, 19), (18, 26), (19, 20), (20, 25), (21, 22), (22, 23), (23, 24), (24, 25)]


In [48]:
node_to_lines = {node: [] for node in range(1, 27)}

for line_index, (a, b) in enumerate(lines):
    node_to_lines[a].append(line_index)
    node_to_lines[b].append(line_index)

print(node_to_lines)

{1: [0, 1, 2], 2: [0, 3, 4, 5, 6], 3: [7], 4: [3], 5: [4, 8], 6: [8, 9], 7: [10, 11], 8: [9], 9: [10, 12], 10: [12], 11: [7, 13], 12: [14, 15], 13: [14, 16], 14: [17], 15: [15], 16: [1], 17: [18], 18: [19, 20], 19: [5, 13, 18, 19, 21], 20: [16, 17, 21, 22], 21: [6, 23], 22: [11, 23, 24], 23: [24, 25], 24: [25, 26], 25: [22, 26], 26: [2, 20]}


In [ ]:
import numpy as np

def build_junction_matrix(line_indices, z_5, zi_index, zt_value):
    n = len(line_indices)
    S = np.zeros((len(line_indices), len(line_indices)), dtype=complex)
    
    for i, line in enumerate(line_indices):
        if line == zi_index:
            # Reflection for incoming line
            S[i, i] = (z_5[zi_index] - (n-1) * zt_value) / (z_5[zi_index] + (n-1) * zt_value)
        else:
            # Transmission to outgoing lines
            S[i, np.where(np.array(line_indices) == zi_index)[0][0]] = 2 * z_5[zi_index] / (z_5[zi_index] + (n-1) * zt_value)
            S[i, i] = 0  # optional: outgoing-to-outgoing reflections

    return S



In [ ]:
import numpy as np

# Example z_5 array (line impedances)
z_5 = np.array(z_5, dtype=complex).flatten()

junction_matrices = {}

for bus in range(1, 27):  # node_numbers
    line_set = node_to_lines[bus]  # get lines connected to this node
    
    if len(line_set) <= 1:
        continue  # no junction, skip
    
    zi_index = line_set[0]        # pick first line as incoming
    outgoing_lines = line_set[1:] # rest are outgoing
    
    if len(outgoing_lines) == 0:
        continue  # no outgoing lines, skip

    # Get Zt from z_5 for outgoing lines
    zt_value = np.mean(z_5[outgoing_lines])  # mean if multiple outgoing lines

    # Build S-matrix using Zi and Zt
    S = build_junction_matrix(line_set, z_5, zi_index, zt_value)
    junction_matrices[bus] = S

# Now junction_matrices[bus] has the S-matrix for each node


In [51]:
print(junction_matrices[1])

[[-0.35733605+0.00151707j  0.        +0.j          0.        +0.j        ]
 [ 0.64266395+0.00151707j  0.        +0.j          0.        +0.j        ]
 [ 0.64266395+0.00151707j  0.        +0.j          0.        +0.j        ]]
